# Notebook 04 — Chat Orchestrator: Intake → JobProfile → SymptomExploration

Drives the LangGraph chat orchestrator through its first three phases against a scripted persona. The persona is an Army 15T (UH-60 crew chief) who served 2005–2014, deployed to Iraq, separated Honorable. The chat collects intake info, validates his MOS against the spine, uses the noise-exposure + risk overlays to prioritise anatomies, and explores hearing in detail (full Baseline + Flare-up + Functional Loss Probes), then asks about back and knee.

**Prereqs:** `make up` (Neo4j), `OPENAI_API_KEY` in `.env`, and slices 5+6 already ingested (CFR §4.71a, the Duty MOS Noise xlsx, and the `mos_risk.yaml` overlay).

The orchestrator is **scripted** for v1 — it consumes a list of veteran responses and runs end-to-end without interactive IO. A future iteration replaces the queue with LangGraph's `interrupt()` to pause for live input.

## 1. Load Concept nodes into the graph

The Concept surface raises inline education the first time a triggering keyword appears in the conversation. The Concepts also persist as `:CFR:Concept` nodes for `/explain` retrieval and later traversal.

In [ ]:
from va_agent.agent.concepts import ingest_concepts, load_concepts
from va_agent.graph.driver import GraphDriver

driver = GraphDriver.from_env()
concepts = load_concepts()
n = ingest_concepts(driver, concepts)
print(f'Ingested {n} Concept nodes:')
for c in concepts:
    print(f'  - {c.id} — {c.name}')

## 2. Run the scripted session

The persona's responses are listed in order. Watch the transcript: the **Worst-Day Rule** Concept should surface inline the first time the chat references a flare-up. Functional Loss Probes are asked one at a time for each anatomy; a "no" answer to a probe (= the activity is something the veteran can't do) gets recorded as a Functional Loss item.

In [ ]:
from va_agent.agent.orchestrator import run_scripted_session
from va_agent.graph.tools import reset_user

USER_ID = 'notebook-04-army-15t'
reset_user(driver, USER_ID)

persona_inputs = [
    # Intake
    'I served in the Army from 2005 to 2014.',
    'Iraq, two tours',
    'Honorable',
    # JobProfile
    'My MOS was 15T (UH-60 crew chief)',
    # SymptomExploration — hearing first (Highly Probable noise exposure)
    'yes',
    'constant ringing in both ears since separation; I miss words in conversations',
    'moderate',
    'severe',
    'every day',
    'most of the day',
    # 4 hearing probes
    'no',   # cannot follow conversation in noisy restaurant
    'yes',  # people do not repeat themselves (not a loss)
    'yes',  # yes I have ringing
    'no',   # TV is too loud (counts as functional loss via the "do you" probe wording)
    # Next anatomy in queue (sorted alphabetically among RISK_FOR): back — say no for this demo
    'no',
    # Next: knee — say no for this demo
    'no',
]

final = run_scripted_session(driver, USER_ID, persona_inputs, concepts=concepts)

## 3. Inspect the transcript

Note how the agent's questions adapt: after the job code is validated, the prioritised-anatomy list is read out (noise exposure first, then the curated RISK_FOR list). The Worst-Day Rule concept appears mid-flow.

In [ ]:
for msg in final['transcript']:
    role = msg['role']
    text = msg['text']
    if role == 'agent':
        print(f'\n🤖  {text}')
    elif role == 'veteran':
        print(f'🧑   {text}')
    else:
        print(f'(system) {text}')

## 4. Inspect the resulting User subgraph

What got persisted as graph nodes for this veteran:

In [ ]:
print('Veteran:')
rows = driver.user_read(
    USER_ID,
    'MATCH (v:User:Veteran {user_id: $user_id}) RETURN v.branch AS branch, v.discharge_characterization AS disc'
)
for r in rows:
    print(f'  branch={r["branch"]}, discharge={r["disc"]}')

print('\nDeployments:')
rows = driver.user_read(
    USER_ID,
    'MATCH (v:User:Veteran {user_id: $user_id})-[:DEPLOYED_TO]->(d:User:Deployment) RETURN d.name AS name'
)
for r in rows:
    print(f'  - {r["name"]}')

print('\nJob Code (with risk profile):')
rows = driver.user_read(
    USER_ID,
    '''MATCH (v:User:Veteran {user_id: $user_id})-[:HOLDS_JOBCODE]->(jc:CFR:JobCode)
       RETURN jc.code AS code, jc.branch AS branch, jc.title AS title'''
)
for r in rows:
    print(f'  {r["code"]} ({r["branch"]}) — {r["title"]}')

print('\nSymptomReports:')
rows = driver.user_read(
    USER_ID,
    '''MATCH (v:User:Veteran {user_id: $user_id})-[:REPORTED]->(sr:User:SymptomReport)
       RETURN sr.body_part AS body_part, sr.text AS text,
              sr.typical_severity AS baseline, sr.flareup_severity AS flare,
              sr.flareup_frequency AS freq, sr.flareup_duration AS dur,
              sr.functional_loss AS losses'''
)
for r in rows:
    print(f'  [{r["body_part"]}] {r["text"]}')
    print(f'      baseline={r["baseline"]}, flare-up={r["flare"]} ({r["freq"]}, lasting {r["dur"]})')
    print(f'      functional_loss={r["losses"]}')

## 5. `/explain` a topic

The `/explain` slash-command lets a veteran ask for a focused explanation on demand. Behind the scenes it does a simple match against the loaded Concepts.

In [ ]:
from va_agent.agent.concepts import explain

for topic in ['pyramiding', 'Bilateral Factor', 'Worst-Day Rule', 'C&P Exam']:
    c = explain(concepts, topic)
    if c:
        print(f'\n=== {c.name} ({c.citation}) ===')
        print(c.plain_language)
    else:
        print(f'\n(no concept found for: {topic})')

## 6. Cleanup

In [ ]:
reset_user(driver, USER_ID)
driver.close()
print('Done.')